## v34 — сохранённый N1/1800 в полной системе

**Run All**, без ограничения по времени. Нового обучения здесь нет: сохраняем конкретные веса v33 (`aux=0.025`) и сравниваем пять фиксированных систем на исходных calibration/validation. Активный v25 не меняется.

Выбор N0/1600 и N1/1800 сделан после inner-анализа. Это сравнение готовых моделей, а не новый независимый тест и не причинная оценка NiVe при одинаковом числе обновлений. Сначала подтверждаются экспорт и прежние итоговые inner graph top-10; затем считаются исходные splits. Все отказы остаются от v25. Изменения диагностического raw пары сохраняются отдельно: этот raw не используется как кандидат.

In [1]:
import os
os.environ['ORT_DISABLE_TELEMETRY'] = '1'  # До импорта ONNX Runtime; macOS SIGABRT workaround.

from pathlib import Path
import sys
import subprocess
import time

HERE = Path.cwd().resolve()
ROOT = next((p for parent in [HERE, *HERE.parents]
             for p in (parent, parent / 'Car-classification-MSK')
             if (p / 'training/nive_system.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Открой notebook внутри Car-classification-MSK')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUN_NAME = 'system_v1'  # Повторный Run All проверит и продолжит этот же запуск.
STARTED = time.perf_counter()
print('Python:', sys.executable)
print('Root:', ROOT)
print('Stage 1/4: component checks. Training: OFF; provider: CPU; time limit: NONE')

Python: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
Root: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK
Stage 1/4: component checks. Training: OFF; provider: CPU; time limit: NONE


In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q',
    'tests/test_nive_system.py', 'tests/test_nive_mixed.py', 'tests/test_nive_low_aux.py',
    'tests/test_dual_role_inference.py', 'tests/test_map_search.py',
    # Сохранённый результат v33 не очищаем ради старого теста пустого notebook.
    '-k', 'not test_notebook_has_clean_run_all_and_new_entrypoint'], check=True, cwd=ROOT)

........................................................................ [ 69%]
................................                                         [100%]
104 passed, 1 deselected in 11.27s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_nive_system.py', 'tests/test_nive_mixed.py', 'tests/test_nive_low_aux.py', 'tests/test_dual_role_inference.py', 'tests/test_map_search.py', '-k', 'not test_notebook_has_clean_run_all_and_new_entrypoint'], returncode=0)

### Фиксация входов

Проверяются исходные checkpoints, изображения, bbox, разбиения, предыдущие результаты и активный v25. `N1` экспортируется **с шага 1800**, не из автоматического экспорта v33 (там выбран step0). Результаты пишутся только в новый каталог v34. Содержимое старых запусков не изменяется.

In [3]:
from training import nive_system as experiment

print('Stage 2/4: read-only preflight and frozen manifest')
context = experiment.prepare(RUN_NAME)
print('Output:', context['output'])
print('Signature:', context['signature'])
print('Frozen threshold:', context['manifest']['threshold'])
for name, checkpoint in context['manifest']['models'].items():
    print(name, 'step=', checkpoint['step'], 'sha256=', checkpoint['sha256'])
print('Optimizer updates: 0; BN updates: 0; automatic promotion: False')

Stage 2/4: read-only preflight and frozen manifest
PREFLIGHT: verify v33 checkpoints, v25 and original splits (read-only)


NiVe integrity: train:   0%|          | 0/17070 [00:00<?, ?it/s]

NiVe integrity: test/query:   0%|          | 0/2887 [00:00<?, ?it/s]

NiVe integrity: test/gallery:   0%|          | 0/11778 [00:00<?, ?it/s]

Output: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_34_nive_system/runs/system_v1
Signature: 67de8e3832c009820492fa1d2ebd3513246d56d5bad9871ec17bbeb2806351b6
Frozen threshold: 0.534365177154541
parent step= 0 sha256= 53f870ea3240c091f9087c5c8db38d83831198a63ff50be2d8741d4e8ac5abd2
N0 step= 1600 sha256= 2f834e1f4216677148ebc12a3a8cbd4ee7196bbeea800b19549f952cdd94316e
N1 step= 1800 sha256= 42bce9b71e1fdcac8981a013670ed5cd3e60210799a818fe5a1ae70f12693a1b
Optimizer updates: 0; BN updates: 0; automatic promotion: False


### Экспорт, inner parity, затем пять систем × два исходных split

- `V25_control`: 50% MVP + 50% прежнего R1 equal3, граф 20/3/0.50.
- `R1_N0_pair` / `R1_N1_pair`: 50% fold-matched parent + 50% N0/1600 или N1/1800, граф 20/3/0.75.
- `MVP_R1_N0` / `MVP_R1_N1`: 50% MVP + 25% parent + 25% N0 или N1, граф 20/3/0.50.

Во всех вариантах candidate/confidence/отказ — прежний R1 equal3 с его замороженным порогом. `raw` в per-query диагностике относится к этому candidate-блоку, а не к новой паре.

Исходная validation используется только для оценки; её изображения, метки и bbox не исправляются. Переиспользованный результат помечается `verified saved result`, извлечение — `fresh` или `cache`. Полный повтор пересчитает только незавершённые этапы. При несовпадении хеша/решений выполнение остановится — не меняй допуски или порог для продолжения.

In [4]:
print('Stage 3/4: export selected weights; parity; original development comparison')
results = experiment.run(context, allow_outer=True)
assert results['status'] == 'complete'
assert results['promoted'] is False and results['candidate_unchanged'] is True

Stage 3/4: export selected weights; parity; original development comparison


/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:2855: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'instance_norm' is set to train=True. Exporting with train=True.
  symbolic_helper.check_training_mode(use_input_stats, "instance_norm")


EXPORT parent: step 0, head-free 512D
EXPORT N0: step 1600, head-free 512D
EXPORT N1: step 1800, head-free 512D
STAGE inner_parity: start
STAGE features_inner_parent: start
EXTRACT inner/parent: 128/1036 | fresh
EXTRACT inner/parent: 256/1036 | fresh
EXTRACT inner/parent: 384/1036 | fresh
EXTRACT inner/parent: 512/1036 | fresh
EXTRACT inner/parent: 640/1036 | fresh
EXTRACT inner/parent: 768/1036 | fresh
EXTRACT inner/parent: 896/1036 | fresh
EXTRACT inner/parent: 1024/1036 | fresh
EXTRACT inner/parent: 1036/1036 | fresh
STAGE features_inner_parent: complete in 40.0s
STAGE features_inner_N0: start
EXTRACT inner/N0: 128/1036 | fresh
EXTRACT inner/N0: 256/1036 | fresh
EXTRACT inner/N0: 384/1036 | fresh
EXTRACT inner/N0: 512/1036 | fresh
EXTRACT inner/N0: 640/1036 | fresh
EXTRACT inner/N0: 768/1036 | fresh
EXTRACT inner/N0: 896/1036 | fresh
EXTRACT inner/N0: 1024/1036 | fresh
EXTRACT inner/N0: 1036/1036 | fresh
STAGE features_inner_N0: complete in 40.5s
STAGE features_inner_N1: start
EXTRA

In [5]:
from IPython.display import Markdown, display

print('Stage 4/4: report')
print(f'This notebook invocation: {(time.perf_counter()-STARTED)/60:.1f} min (not a benchmark)')
print('Report:', context['output'] / 'REPORT.md')
display(Markdown((context['output'] / 'REPORT.md').read_text(encoding='utf-8')))
print('Готово. v25 не переключён. Можно прислать сообщение: подсчёты завершились.')

Stage 4/4: report
This notebook invocation: 8.0 min (not a benchmark)
Report: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_34_nive_system/runs/system_v1/REPORT.md


# v34 — сохранённая NiVe-смесь против v25

Обучения, подбора порога и переключения MVP не было.
Сохранённые parent/N0/N1 обучались на 740 train-ID; эти ID не пересекаются с original calibration/validation.
v25 использует собственный full-train ансамбль. Сравниваются готовые системы, не одинаковый объём обучения.

| Split | Система | mAP@10 | Δ к v25, п.п. | Rank-1 | F1 | TNR |
|---|---|---:|---:|---:|---:|---:|
| calibration | V25_control | 0.826410 | +0.0000 | 0.841463 | 0.813483 | 0.704918 |
| calibration | R1_N0_pair | 0.810147 | -1.6263 | 0.821138 | 0.813483 | 0.704918 |
| calibration | R1_N1_pair | 0.805896 | -2.0514 | 0.817073 | 0.813483 | 0.704918 |
| calibration | MVP_R1_N0 | 0.810438 | -1.5972 | 0.829268 | 0.813483 | 0.704918 |
| calibration | MVP_R1_N1 | 0.805309 | -2.1101 | 0.821138 | 0.813483 | 0.704918 |
| validation | V25_control | 0.828957 | +0.0000 | 0.829960 | 0.795455 | 0.709677 |
| validation | R1_N0_pair | 0.801491 | -2.7467 | 0.793522 | 0.795455 | 0.709677 |
| validation | R1_N1_pair | 0.806967 | -2.1990 | 0.801619 | 0.795455 | 0.709677 |
| validation | MVP_R1_N0 | 0.823858 | -0.5099 | 0.825911 | 0.795455 | 0.709677 |
| validation | MVP_R1_N1 | 0.828912 | -0.0045 | 0.829960 | 0.795455 | 0.709677 |

Пять конфигураций зафиксированы до этого запуска; нет выбора по calibration перед просмотром validation.
Inner graph top-10 совпали точно; изменения диагностического raw пары: {'N0': 0, 'N1': 1}.
Эти raw-списки не используются для candidate/отказа. Полные расхождения сохранены в results.json; это не заявление о побитовой переносимости всех внутренних оценок.
Исходная validation уже многократно использовалась: это development, не независимый тест.
N0/1600 и N1/1800 выбраны после inner-анализа: это контроль конкретных кандидатов, не причинная оценка NiVe при равном числе обновлений.
R1_N* — равная пара с графом 20/3/.75; MVP_R1_N* — .50 MVP + .25 parent + .25 N* с графом 20/3/.50.
Кандидат/уверенность/отказ всегда от прежнего full-train R1 equal3 и его порога. CSV кандидатов совпадает побайтно.
3072D export — реальные блоки MVP512 + candidate1536 + parent512 + member512; это не один cosine ranking-vector.
NPY replay использует явные slices/веса. Не утверждается, что вспомогательная mINP raw-bank совпадает с graph mAP.
У полной новой системы 6 encoder forwards вместо 4 у v25: сохранён старый candidate-ансамбль. GPU/скорость не подтверждены.
Номерная устойчивость не доказана; OCR/обработка номеров не добавлялись. Junk/top10 и evaluator не менялись.
Для нового профиля/замены MVP требуется отдельное решение. Старые v32/v33 отчёты не переписаны.


Готово. v25 не переключён. Можно прислать сообщение: подсчёты завершились.
